In [0]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 31.5204,
    "longitude": 74.3587,
    "current": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"],
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json())

200
{'latitude': 31.528997, 'longitude': 74.38995, 'generationtime_ms': 0.11432170867919922, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 220.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-10-12T10:30', 'interval': 900, 'temperature_2m': 27.6, 'relative_humidity_2m': 53, 'wind_speed_10m': 3.5}}


In [0]:

import requests

cities = {
    "Lahore": {"latitude": 31.5204, "longitude": 74.3587},
    "Karachi": {"latitude": 24.8607, "longitude": 67.0011},
    "Islamabad": {"latitude": 33.6844, "longitude": 73.0479}
}

url = "https://api.open-meteo.com/v1/forecast"

weather_data = []

for city, location in cities.items():
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m"
        ]
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    weather = response.json()["current"]

    record = {
        "city": city,
        "temperature": weather["temperature_2m"],
        "humidity": weather["relative_humidity_2m"],
        "wind_speed": weather["wind_speed_10m"],
        "time": weather["time"]
    }

    weather_data.append(record)

for record in weather_data:
    print(record)


{'city': 'Lahore', 'temperature': 27.6, 'humidity': 53, 'wind_speed': 3.5, 'time': '2026-10-12T10:30'}
{'city': 'Karachi', 'temperature': 31.4, 'humidity': 60, 'wind_speed': 13.5, 'time': '2026-10-12T10:30'}
{'city': 'Islamabad', 'temperature': 27.0, 'humidity': 50, 'wind_speed': 2.4, 'time': '2026-10-12T10:30'}


In [0]:
%python
print("Weather Data Pipeline Started!")


Weather Data Pipeline Started!


In [0]:

import requests
import pandas as pd
from datetime import date, timedelta

cities = {
    "Lahore": (31.5204, 74.3587),
    "Karachi": (24.8607, 67.0011),
    "Islamabad": (33.6844, 73.0479)
}

end_date = date.today() - timedelta(days=1)
start_date = end_date - timedelta(days=29)

url = "https://archive-api.open-meteo.com/v1/archive"

all_records = []

for city, (latitude, longitude) in cities.items():
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m"
        ],
        "timezone": "Asia/Karachi"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()["hourly"]

    for i, timestamp in enumerate(data["time"]):
        all_records.append({
            "city": city,
            "time": timestamp,
            "temperature": data["temperature_2m"][i],
            "humidity": data["relative_humidity_2m"][i],
            "wind_speed": data["wind_speed_10m"][i]
        })

weather_df = pd.DataFrame(all_records)

print("Start date:", start_date)
print("End date:", end_date)
print("Total records:", len(weather_df))

display(weather_df)


Start date: 2026-09-12
End date: 2026-10-11
Total records: 2160


city,time,temperature,humidity,wind_speed
Lahore,2026-09-12T00:00,29.7,64,10.0
Lahore,2026-09-12T01:00,29.1,71,10.6
Lahore,2026-09-12T02:00,28.6,76,9.6
Lahore,2026-09-12T03:00,28.1,80,7.8
Lahore,2026-09-12T04:00,27.7,83,6.1
Lahore,2026-09-12T05:00,27.9,83,6.0
Lahore,2026-09-12T06:00,27.3,85,5.5
Lahore,2026-09-12T07:00,28.1,81,5.6
Lahore,2026-09-12T08:00,29.7,74,5.6
Lahore,2026-09-12T09:00,31.5,66,5.9


In [0]:
from pyspark.sql import functions as F

# Pandas DataFrame ko Spark DataFrame mein convert karna
bronze_df = spark.createDataFrame(weather_df)

# Data ingestion ka time add karna
bronze_df = bronze_df.withColumn(
    "ingestion_time",
    F.current_timestamp()
)

# Pehli 5 rows dekhna
display(bronze_df.limit(5))

city,time,temperature,humidity,wind_speed,ingestion_time
Lahore,2026-09-12T00:00,29.7,64,10.0,2026-10-12T10:59:12.310664Z
Lahore,2026-09-12T01:00,29.1,71,10.6,2026-10-12T10:59:12.310664Z
Lahore,2026-09-12T02:00,28.6,76,9.6,2026-10-12T10:59:12.310664Z
Lahore,2026-09-12T03:00,28.1,80,7.8,2026-10-12T10:59:12.310664Z
Lahore,2026-09-12T04:00,27.7,83,6.1,2026-10-12T10:59:12.310664Z


In [0]:
# Bronze data ko Delta table mein save karna
(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather_bronze")
)

print("Bronze Layer successfully created!")

Bronze Layer successfully created!


In [0]:
display(spark.table("weather_bronze"))


city,time,temperature,humidity,wind_speed,ingestion_time
Islamabad,2026-09-12T00:00,26.3,79,4.1,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T01:00,25.8,80,3.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T02:00,25.5,80,3.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T03:00,25.3,80,3.7,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T04:00,25.4,79,3.6,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T05:00,25.4,80,6.2,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T06:00,24.7,84,2.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T07:00,25.5,81,3.5,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T08:00,26.6,76,5.0,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-12T09:00,28.3,70,2.4,2026-10-12T11:01:23.333114Z


In [0]:
from pyspark.sql import functions as F

# Bronze table se data read karna
silver_df = spark.table("weather_bronze")

# Missing values wale records remove karna
silver_df = silver_df.dropna(
    subset=["city", "time", "temperature", "humidity", "wind_speed"]
)

# Duplicate records remove karna
silver_df = silver_df.dropDuplicates(["city", "time"])

# Data types ko correct karna
silver_df = (
    silver_df
    .withColumn("temperature", F.col("temperature").cast("double"))
    .withColumn("humidity", F.col("humidity").cast("double"))
    .withColumn("wind_speed", F.col("wind_speed").cast("double"))
    .withColumn("time", F.to_timestamp("time"))
)

# Results check karna
print("Bronze records:", spark.table("weather_bronze").count())
print("Silver records:", silver_df.count())

display(silver_df.limit(10))

Bronze records: 2160
Silver records: 2160


city,time,temperature,humidity,wind_speed,ingestion_time
Islamabad,2026-09-13T02:00:00Z,24.8,82.0,7.7,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-26T14:00:00Z,31.2,36.0,13.0,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-27T12:00:00Z,30.4,47.0,4.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-10-01T23:00:00Z,24.4,69.0,2.6,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-25T09:00:00Z,23.0,77.0,5.4,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-29T06:00:00Z,19.3,83.0,3.8,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-14T09:00:00Z,23.0,90.0,6.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-22T13:00:00Z,31.7,47.0,3.1,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-30T12:00:00Z,30.2,39.0,8.5,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-24T14:00:00Z,29.3,55.0,6.1,2026-10-12T11:01:23.333114Z


In [0]:
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather_silver")
)

print("Silver Layer successfully created!")

Silver Layer successfully created!


In [0]:
display(spark.table("weather_silver"))

city,time,temperature,humidity,wind_speed,ingestion_time
Islamabad,2026-09-13T02:00:00Z,24.8,82.0,7.7,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-26T14:00:00Z,31.2,36.0,13.0,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-27T12:00:00Z,30.4,47.0,4.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-10-01T23:00:00Z,24.4,69.0,2.6,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-25T09:00:00Z,23.0,77.0,5.4,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-29T06:00:00Z,19.3,83.0,3.8,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-14T09:00:00Z,23.0,90.0,6.9,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-22T13:00:00Z,31.7,47.0,3.1,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-30T12:00:00Z,30.2,39.0,8.5,2026-10-12T11:01:23.333114Z
Islamabad,2026-09-24T14:00:00Z,29.3,55.0,6.1,2026-10-12T11:01:23.333114Z


In [0]:
from pyspark.sql import functions as F

# Silver table se data read karna
silver_df = spark.table("weather_silver")

# Har city ka weather summary
gold_df = (
    silver_df
    .groupBy("city")
    .agg(
        F.round(F.avg("temperature"), 2).alias("avg_temperature"),
        F.round(F.avg("humidity"), 2).alias("avg_humidity"),
        F.round(F.avg("wind_speed"), 2).alias("avg_wind_speed"),
        F.count("*").alias("total_records")
    )
)

display(gold_df)

city,avg_temperature,avg_humidity,avg_wind_speed,total_records
Karachi,28.76,75.17,11.2,720
Lahore,28.23,71.36,5.18,720
Islamabad,25.44,68.6,5.09,720


In [0]:
(gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather_gold")
)

print("Gold Layer successfully created!")


Gold Layer successfully created!


In [0]:
tables = [
    "weather_bronze",
    "weather_silver",
    "weather_gold"
]

for table_name in tables:
    df = spark.table(table_name)
    print(table_name, "=", df.count(), "records")

weather_bronze = 2160 records
weather_silver = 2160 records
weather_gold = 3 records


In [0]:
from pyspark.sql import functions as F

df = spark.table("weather_silver")

# Missing values check
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["city", "time", "temperature", "humidity", "wind_speed"]
]).show()

# Duplicate records check
print("Duplicate records:", 
      df.count() - df.dropDuplicates(["city", "time"]).count())

# Weather values check
df.select(
    F.min("temperature").alias("min_temp"),
    F.max("temperature").alias("max_temp"),
    F.min("humidity").alias("min_humidity"),
    F.max("humidity").alias("max_humidity"),
    F.min("wind_speed").alias("min_wind"),
    F.max("wind_speed").alias("max_wind")
).show()

+----+----+-----------+--------+----------+
|city|time|temperature|humidity|wind_speed|
+----+----+-----------+--------+----------+
|   0|   0|          0|       0|         0|
+----+----+-----------+--------+----------+

Duplicate records: 0
+--------+--------+------------+------------+--------+--------+
|min_temp|max_temp|min_humidity|max_humidity|min_wind|max_wind|
+--------+--------+------------+------------+--------+--------+
|    18.0|    37.2|        35.0|       100.0|     0.0|    27.0|
+--------+--------+------------+------------+--------+--------+



In [0]:
gold_df = spark.table("weather_gold")

display(gold_df)

city,avg_temperature,avg_humidity,avg_wind_speed,total_records
Karachi,28.76,75.17,11.2,720
Lahore,28.23,71.36,5.18,720
Islamabad,25.44,68.6,5.09,720


In [0]:
gold_df = spark.table("weather_gold")

gold_pdf = gold_df.toPandas()

gold_pdf.to_csv("/tmp/weather_gold.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [0]:
display(gold_df)

city,avg_temperature,avg_humidity,avg_wind_speed,total_records
Karachi,28.76,75.17,11.2,720
Lahore,28.23,71.36,5.18,720
Islamabad,25.44,68.6,5.09,720
